# 12 — BibTeX extraction and reference deduplication (Milestone 2)

Owner: **Member 3**. Definitions only, like notebooks 01–04: `13_dedup_export` / `15_run_parsing`
load it with `%run -i ./12_bibtex.ipynb` (with `RUN_TEST = False`) and call the functions below.

What this notebook does for every publication:

1. **Extract** the reference entries of each version from one of three sources
   (`.bib` declared by `\bibliography` / `\addbibresource`, `\bibitem` in the gathered `.tex`,
   `\bibitem` or biblatex records in a `.bbl`), and convert `\bibitem` text to BibTeX fields.
2. **Label** the source of each version: `bib`, `bibitem`, `bbl`, `missing_bbl`, `none`.
3. **Deduplicate** entries inside a version and across versions (DOI, arXiv id, or
   normalized title + year), **choose one key** (the newest version's key), **union the fields**,
   and return a `key_map` of renamed keys.
4. **Rename** `\cite`-family keys with that map (`rename_cites`).
5. **Export** the `bibtex` table (9 columns of the statement) plus `key_map` and per-version stats.

| Function | Returns |
|---|---|
| `parse_bib(text)` | `list[dict]` entries of a `.bib` file |
| `parse_bibitems(text)` | `list[dict]` entries converted from `\bibitem` (in `.tex` or `.bbl`) |
| `parse_biblatex_bbl(text)` | `list[dict]` entries of a biblatex `.bbl` |
| `extract_version_refs(version_dir, tex)` | `VersionRefs` (source label, entries, counts) |
| `dedup_references({version: entries})` | `DedupResult` (unified entries, `key_map`, stats) |
| `cited_keys(text)` | `Counter` of the keys cited in a text |
| `rename_cites(text, mapping)` | text with renamed cite keys |
| `process_paper_bib(arxiv_id, {version: (dir, tex)})` | everything above for one paper |
| `bibtex_rows`, `key_map_rows`, `entry_to_bibtex` | table rows / BibTeX text |
| `build_bibtex_tables(data_dir, out_dir, limit=None)` | runs every paper folder, writes the tables |
| `bibtex_summary(tables, errors, seconds)` | the run numbers reported in the book |

Only the standard library is needed (pandas / pyarrow are optional, for the table export).

In [ ]:
import json
import re
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

if "RUN_TEST" not in globals():
    RUN_TEST = True
if "CONFIG_LOADED" not in globals():      # DATA_DIR / PROJECT_ROOT for the standalone run (section 12)
    try:
        get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
    except Exception as e:                 # plain Python / tests without the repo: helpers still work
        print(f"[bibtex] config not loaded ({type(e).__name__}); offline functions only")

# Keep only entries that the version actually cites (BibTeX itself prints only cited entries,
# so this mirrors the compiled PDF). `\nocite{*}` or a version without any \cite keeps all.
BIB_ONLY_CITED = True

# Columns of the `bibtex` table required by the statement (Milestone 2, Requirement)
BIBTEX_COLUMNS = ["arxiv_id", "citation_key", "entry_type", "title", "authors",
                  "year", "doi", "journal", "url"]
SOURCE_LABELS = ("bib", "bibitem", "bbl", "missing_bbl", "none")

## 1. LaTeX → plain text for field values

Field values keep LaTeX markup (`M{\"u}nster`, `{\em Physical Review E}`, `Lincoln~F.`).
`latex_to_text` turns them into readable Unicode text; `norm_title` builds the matching key used
by deduplication (accents removed, lower case, only letters and digits).

In [ ]:
_ACCENTS = {  # LaTeX accent command -> Unicode combining mark
    '"': "\u0308", "'": "\u0301", "`": "\u0300", "^": "\u0302", "~": "\u0303",
    "=": "\u0304", ".": "\u0307", "u": "\u0306", "v": "\u030c", "H": "\u030b",
    "c": "\u0327", "k": "\u0328", "r": "\u030a", "d": "\u0323", "b": "\u0331",
}
_SPECIAL = {  # letter-like commands
    "ss": "ß", "o": "ø", "O": "Ø", "aa": "å", "AA": "Å", "ae": "æ", "AE": "Æ",
    "oe": "œ", "OE": "Œ", "l": "ł", "L": "Ł", "i": "ı", "j": "ȷ",
    "&": "&", "%": "%", "$": "$", "#": "#", "_": "_", "{": "{", "}": "}",
    "textendash": "–", "textemdash": "—", "textquoteright": "'", "textquoteleft": "'",
    "ldots": "...", "dots": "...", "textregistered": "®", "copyright": "©", "S": "§",
}
_ACCENT_RE = re.compile(r"""\\(["'`^~=.]|[uvHckrdb](?![A-Za-z]))\s*(?:\{\s*(\\?[A-Za-z]|)\s*\}|(\\?[A-Za-z]))""")
_SPECIAL_RE = re.compile(r"\\(ss|aa|AA|ae|AE|oe|OE|[oOlLij])(?![A-Za-z])\s*(?:\{\})?|"
                         r"\\(textendash|textemdash|textquoteright|textquoteleft|ldots|dots|"
                         r"textregistered|copyright|S)(?![A-Za-z])\s*(?:\{\})?|\\([&%$#_{}])")
# Commands whose argument is kept as text (font changes, links, ...)
_KEEP_ARG_RE = re.compile(r"\\(?:emph|textit|textbf|textsc|textrm|textsf|texttt|textup|textmd|"
                          r"textnormal|mbox|hbox|text|mathrm|mathit|mathbf|mathsf|mathtt|mathcal|"
                          r"url|path|nolinkurl|uppercase|lowercase|MakeUppercase|MakeLowercase|"
                          r"bibinfo\s*\{[^{}]*\}|bibfield\s*\{[^{}]*\}|bibnamefont|bibfnamefont|"
                          r"citenamefont|natexlab|enquote|textsuperscript|ensuremath)\s*\{")
_FONT_SWITCH_RE = re.compile(r"\\(?:em|it|bf|sc|rm|sf|tt|sl|small|footnotesize|scriptsize|"
                             r"normalfont|itshape|bfseries|scshape|upshape|mdseries|"
                             r"relax|protect|newblock|noopsort\s*\{[^{}]*\}|/)(?![A-Za-z])\s*")


def _matching_brace(text, open_pos):
    """Index of the '}' that closes the '{' at open_pos (or -1). Escaped braces are skipped."""
    depth = 0
    i, n = open_pos, len(text)
    while i < n:
        ch = text[i]
        if ch == "\\":
            i += 2
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return i
        i += 1
    return -1


def _unwrap_commands(text, pattern):
    """Replace `\\cmd{arg}` by `arg` for every command matched by `pattern` (nested-safe)."""
    for _ in range(20):                      # nested commands: repeat until nothing changes
        m = pattern.search(text)
        if not m:
            break
        out, pos = [], 0
        for m in pattern.finditer(text):
            if m.start() < pos:
                continue
            close = _matching_brace(text, m.end() - 1)
            if close < 0:
                continue
            out.append(text[pos:m.start()])
            out.append(text[m.end():close])
            pos = close + 1
        out.append(text[pos:])
        new = "".join(out)
        if new == text:
            break
        text = new
    return text


def latex_to_text(value):
    """'M{\\"u}nster, {\\em Phys. Rev.}~E' -> 'Münster, Phys. Rev. E' (math `$..$` is kept)."""
    if not value:
        return ""
    s = str(value)
    s = re.sub(r"(?<!\\)%[^\n]*", "", s)                  # comments
    s = re.sub(r"\\href\s*\{[^{}]*\}\s*", "", s)           # \href{url}{text} -> {text}
    s = re.sub(r"\\\((.*?)\\\)", r"$\1$", s)              # \( x \) -> $x$ (same as math normalization)
    s = _ACCENT_RE.sub(lambda m: unicodedata.normalize(
        "NFC", (m.group(2) or m.group(3) or "").lstrip("\\").replace("i", "i") + _ACCENTS[m.group(1)])
        if (m.group(2) or m.group(3)) else "", s)
    s = _SPECIAL_RE.sub(lambda m: _SPECIAL.get(m.group(1) or m.group(2) or m.group(3), ""), s)
    s = _unwrap_commands(s, _KEEP_ARG_RE)
    s = _FONT_SWITCH_RE.sub("", s)
    s = re.sub(r"\\(?:[,;:! \n]|(?:quad|qquad|thinspace|enspace|xspace)(?![A-Za-z]))", " ", s)
    s = s.replace("~", " ").replace("``", '"').replace("''", '"').replace("---", "—").replace("--", "–")
    s = re.sub(r"\\[A-Za-z]+\*?(?=\s*\{)", "", s)          # unknown \cmd{x} -> {x}
    s = re.sub(r"(?<!\\)[{}]", "", s)                      # grouping braces
    s = re.sub(r"\\(?=[A-Za-z])", "", s) if "$" not in s else s
    s = re.sub(r"\s+", " ", s).strip()
    return s.strip(" ,;")


def _ascii_fold(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")


def norm_title(title):
    """Matching key for titles: accents and LaTeX removed, lower case, [a-z0-9] tokens only."""
    if not title:
        return ""
    s = _ascii_fold(latex_to_text(title)).lower()
    s = re.sub(r"\$[^$]*\$", lambda m: re.sub(r"\\[A-Za-z]+", " ", m.group(0)), s)
    return " ".join(re.findall(r"[a-z0-9]+", s))


def surname_key(author):
    """'Lambert Münster' / 'Münster, Lambert' -> 'munster' (used as a tie-breaker for short titles)."""
    if not author:
        return ""
    a = _ascii_fold(author).lower()
    a = a.split(",")[0] if "," in a else (a.split()[-1] if a.split() else "")
    return re.sub(r"[^a-z]", "", a)

## 2. Identifiers inside an entry: DOI, arXiv id, year

These are the strongest deduplication keys, so they are read from every field where authors
usually put them (`doi`, `eprint`, `url`, `journal = {arXiv preprint arXiv:...}`, `note`, ...).

In [ ]:
_DOI_RE = re.compile(r"\b(10\.\d{4,9}/[^\s\"<>{}\\,;]+)", re.I)
_ARXIV_NEW_RE = re.compile(r"(?:arxiv[:\s/.]*|abs/|pdf/|arXiv\.)?\b(\d{4}\.\d{4,5})(?:v\d+)?\b", re.I)
_ARXIV_CTX_RE = re.compile(r"arxiv(?:\.org/(?:abs|pdf)/|[:\s]+|\.)\s*(\d{4}\.\d{4,5}|[a-z\-]+(?:\.[A-Z]{2})?/\d{7})"
                           r"(?:v\d+)?", re.I)
_ARXIV_OLD_RE = re.compile(r"\b([a-z\-]+(?:\.[A-Z]{2})?/\d{7})(?:v\d+)?\b")
_YEAR_RE = re.compile(r"(?<![\d./])((?:19|20)\d{2})(?![\d])")


def norm_doi(value):
    """'https://doi.org/10.1103/PhysRevE.58.2749.' -> '10.1103/physreve.58.2749'."""
    if not value:
        return ""
    m = _DOI_RE.search(str(value).replace("\\_", "_"))
    return m.group(1).rstrip(".").lower() if m else ""


def find_arxiv_id(fields):
    """arXiv id (no version) cited by an entry, or ''. Looks at eprint/arxiv fields, URLs, DOI, notes."""
    eprint = str(fields.get("eprint", "") or fields.get("arxivid", "") or fields.get("arxiv", "")).strip()
    prefix = str(fields.get("archiveprefix", "") or fields.get("eprinttype", "")).lower()
    if eprint:
        m = re.match(r"^(?:arxiv:)?\s*(\d{4}\.\d{4,5}|[a-z\-]+(?:\.[A-Z]{2})?/\d{7})(?:v\d+)?$", eprint, re.I)
        if m and (not prefix or "arxiv" in prefix or re.match(r"\d{4}\.\d{4,5}", m.group(1))):
            return m.group(1)
    doi = norm_doi(fields.get("doi", ""))
    if doi.startswith("10.48550/arxiv."):
        return doi.split("arxiv.", 1)[1]
    for name in ("journal", "url", "note", "howpublished", "booktitle", "publisher", "volume", "_raw"):
        m = _ARXIV_CTX_RE.search(str(fields.get(name, "")))
        if m:
            return m.group(1)
    return ""


def parse_year(value):
    """First plausible year in a field ('2021', '2021-05-03', 'in press, 2019') or None."""
    m = _YEAR_RE.search(str(value or ""))
    return int(m.group(1)) if m else None

## 3. Authors

BibTeX separates authors with ` and `; `Last, First` is rewritten as `First Last`.
Free-text references (`\bibitem`) use commas, `and` or `&`; when names are written as
`Surname, X.` (APA style) the surname and the initials are paired back together.

In [ ]:
def _split_top(text, seps_re):
    """Split on a regex, but only outside braces."""
    parts, depth, last = [], 0, 0
    for m in re.finditer(r"[{}]|" + seps_re, text):
        tok = m.group(0)
        if tok == "{":
            depth += 1
        elif tok == "}":
            depth = max(0, depth - 1)
        elif depth == 0:
            parts.append(text[last:m.start()])
            last = m.end()
    parts.append(text[last:])
    return parts


def _flip_name(name):
    """'Vaswani, Ashish' -> 'Ashish Vaswani'; 'Doe, Jr., John' -> 'John Doe Jr.'."""
    parts = [p.strip() for p in _split_top(name, r",") if p.strip()]
    if len(parts) == 2:
        return f"{parts[1]} {parts[0]}"
    if len(parts) == 3:
        return f"{parts[2]} {parts[0]} {parts[1]}"
    return name


def split_bib_authors(value):
    """BibTeX author field -> list of 'First Last' strings ('others' is dropped)."""
    if not value:
        return []
    out = []
    for name in _split_top(str(value), r"\s+and\s+"):
        name = re.sub(r"\s+", " ", name).strip()
        if not name or name.lower() in ("others", "et al.", "et al"):
            continue
        out.append(latex_to_text(_flip_name(name)))
    return [a for a in out if a]


_INITIALS_RE = re.compile(r"^(?:[A-Z]\.?[\s\-~]*){1,4}$")


def split_free_authors(text):
    """Free-text author list of a \\bibitem -> list of names."""
    s = latex_to_text(text)
    s = re.sub(r"\bet\s*al\.?", "", s)
    s = re.sub(r",?\s+(?:and|&)\s+", ", ", s)
    parts = [p.strip(" .") for p in s.split(",") if p.strip(" .")]
    # APA style "Surname, X., Surname, Y." -> pair surname + initials
    if len(parts) >= 2 and all(_INITIALS_RE.match(p + ".") for p in parts[1::2]) and \
            not any(_INITIALS_RE.match(p + ".") for p in parts[0::2]):
        parts = [f"{parts[i + 1]}. {parts[i]}".replace("..", ".") for i in range(0, len(parts) - 1, 2)]
    return [p for p in parts if 1 < len(p) < 80]

## 4. `.bib` parser

A small brace-aware parser (no dependency): handles `{...}` / `"..."` values, numbers,
`@string` macros, `#` concatenation, `@comment` / `@preamble`, and entries written with `(...)`.
Field names are lower-cased. A broken entry is skipped instead of stopping the whole file.

In [ ]:
_MONTHS = {m: str(i + 1) for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}


def _read_value(text, i, strings):
    """Read one field value starting at text[i]; returns (value, next_index)."""
    pieces, n = [], len(text)
    while True:
        while i < n and text[i].isspace():
            i += 1
        if i >= n:
            break
        ch = text[i]
        if ch == "{":
            close = _matching_brace(text, i)
            if close < 0:
                return "".join(pieces), n
            pieces.append(text[i + 1:close])
            i = close + 1
        elif ch == '"':
            j, depth = i + 1, 0
            while j < n:
                c = text[j]
                if c == "\\":
                    j += 2
                    continue
                if c == "{":
                    depth += 1
                elif c == "}":
                    depth -= 1
                elif c == '"' and depth <= 0:
                    break
                j += 1
            pieces.append(text[i + 1:j])
            i = j + 1
        else:
            m = re.match(r"[^\s,#}\)]+", text[i:])
            if not m:
                break
            word = m.group(0)
            low = word.lower()
            pieces.append(strings[low] if low in strings else _MONTHS.get(low, word))
            i += len(word)
        while i < n and text[i].isspace():
            i += 1
        if i < n and text[i] == "#":
            i += 1
            continue
        break
    return "".join(pieces), i


def parse_bib(text):
    """Parse BibTeX source -> list of {'key', 'type', 'fields'} (fields lower-cased, raw LaTeX values)."""
    entries, strings = [], {}
    i, n = 0, len(text)
    while True:
        at = text.find("@", i)
        if at < 0:
            break
        m = re.match(r"@\s*([A-Za-z]+)\s*([{(])", text[at:])
        if not m:
            i = at + 1
            continue
        etype, opener = m.group(1).lower(), m.group(2)
        start = at + m.end()
        if opener == "{":
            end = _matching_brace(text, start - 1)
        else:                                   # @article( ... ) form
            depth, end = 1, -1
            for j in range(start, n):
                if text[j] == "(":
                    depth += 1
                elif text[j] == ")":
                    depth -= 1
                    if depth == 0:
                        end = j
                        break
        if end < 0:                              # unbalanced: skip to the next '@'
            i = start
            continue
        body = text[start:end]
        i = end + 1
        if etype in ("comment", "preamble", "ieeetranbstctl"):   # not references
            continue
        if etype == "string":
            sm = re.match(r"\s*([A-Za-z][\w\-:.]*)\s*=\s*", body)
            if sm:
                strings[sm.group(1).lower()], _ = _read_value(body, sm.end(), strings)
            continue
        km = re.match(r"\s*([^,\s]+)\s*,", body)
        if not km:
            continue
        key, fields, j = km.group(1).strip(), {}, km.end()
        while j < len(body):
            fm = re.compile(r"\s*([A-Za-z][\w\-:.+]*)\s*=\s*").match(body, j)
            if not fm:
                nxt = body.find(",", j)
                if nxt < 0:
                    break
                j = nxt + 1
                continue
            value, j = _read_value(body, fm.end(), strings)
            fields[fm.group(1).lower()] = re.sub(r"\s+", " ", value).strip()
            while j < len(body) and body[j] in " \t\r\n,":
                j += 1
        entries.append({"key": key, "type": etype, "fields": fields})
    return entries

## 5. `\bibitem` → BibTeX fields (in `.tex` and in classic `.bbl`)

A `\bibitem` holds free text, so its fields are recovered with rules ordered from most to
least reliable:

* identifiers first: DOI (`\doi{}`, `doi.org/...`), URL (`\url{}`, `\href{}`), arXiv id, year;
* **`\newblock` layout** (natbib / BibTeX styles, e.g. `.bbl` files): block 1 = authors,
  block 2 = title, block 3 = venue;
* **quoted title** (IEEE style): ``` ``Title,'' ``` — authors before it, venue after it;
* **comma layout** (elsarticle / math styles): leading name-like segments are authors, the next
  segment is the title, the rest is the venue;
* otherwise **period layout**: `Authors. Title. Venue.`

The venue is the emphasized text (`\emph{}`, `{\em }`, `{\it }`); `In \emph{...}` means a
conference (`booktitle`). Every item keeps its raw text in `_raw`, so nothing is lost.

In [ ]:
_BIBITEM_RE = re.compile(r"\\bibitem\s*(\[(?:[^\[\]]|\[[^\]]*\])*\])?\s*\{([^{}]*)\}")
_NAME_SEG_RE = re.compile(r"^(?:(?:[A-Z][a-z]?\.?[\s\-]*){1,4}(?:(?:van|von|de|der|da|di|del|le|la)\s+)*"
                          r"[A-Z][\w'\-]+(?:\s+[A-Z][\w'\-]+)?|"
                          r"[A-Z][\w'\-]+(?:\s+[A-Z][\w'\-]+)?,?\s+(?:[A-Z]\.[\s\-]*){1,4}|"
                          r"[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}|et al\.?)$")


def _emph_text(text):
    """First emphasized span: \\emph{X}, {\\em X}, {\\it X}, \\textit{X}."""
    m = re.search(r"\\(?:emph|textit)\s*\{", text)
    if m:
        close = _matching_brace(text, m.end() - 1)
        if close > 0:
            return text[m.end():close]
    m = re.search(r"\{\\(?:em|it|itshape|sl)\b\s*", text)
    if m:
        close = _matching_brace(text, m.start())
        if close > 0:
            return text[m.end():close]
    return ""


def _strip_ids(text):
    """Remove DOI / URL / arXiv / pages noise before looking for title and venue."""
    t = re.sub(r"\\(?:doi|url|href)\s*\{[^{}]*\}(?:\s*\{[^{}]*\})?", " ", text)
    t = re.sub(r"(?:https?://|www\.)\S+", " ", t)
    t = re.sub(r"\b(?:doi|DOI)\s*:?\s*10\.\S+", " ", t)
    return t


def _venue_fields(venue_raw, fields):
    """Fill journal / booktitle / volume / pages from the venue part of a free-text reference."""
    if not venue_raw:
        return
    emph = _emph_text(venue_raw)
    in_proc = bool(re.match(r"\s*(?:in|In)\b[:\s]", latex_to_text(venue_raw))) or \
        re.search(r"\bProc(?:eedings|\.)|Conference|Workshop|Symposium|\bICML\b|\bNeurIPS\b|\bCVPR\b",
                  venue_raw)
    plain = re.sub(r"^\s*(?:in|In)\b[:\s]*", "", latex_to_text(venue_raw))
    name = latex_to_text(emph) if emph else re.split(
        r",|\(|\bpp?\.|\bvol\.|\bno\.|\s\d+\s*[(:]|\s(?:19|20)\d{2}\b(?!\s+[A-Z])", plain, maxsplit=1)[0]
    name = re.sub(r"[\s\d.;:]+$", "", name)
    name = name.strip(" .,;")
    if name and not re.fullmatch(r"(?:19|20)\d{2}|pp?\.?|vol\.?", name, re.I):
        if re.search(r"arxiv", name, re.I):
            fields.setdefault("journal", name)
        elif in_proc:
            fields.setdefault("booktitle", name)
        else:
            fields.setdefault("journal", name)
    txt = latex_to_text(venue_raw)
    m = re.search(r"(?:vol\.?\s*|\b)(\d{1,4})\s*(?:\((\d{1,4}[\-–]?\d*)\))?\s*[:,]\s*(?:pp\.?\s*)?(\d+\s*[–\-]+\s*\d+|\d+)", txt)
    if m:
        fields.setdefault("volume", m.group(1))
        fields.setdefault("pages", re.sub(r"\s*[–\-]+\s*", "--", m.group(3)))
    m = re.search(r"pp\.?\s*(\d+\s*[–\-]+\s*\d+)", txt)
    if m:
        fields["pages"] = re.sub(r"\s*[–\-]+\s*", "--", m.group(1))


_PARTICLES = {"van", "von", "de", "der", "den", "da", "di", "del", "della", "du", "le", "la", "dos", "y", "ter"}


def _looks_like_name(part):
    """'H. Cohn', 'Daniel D Johnson', 'Rianne Van Den Berg', 'J. W. H.' -> True; title text -> False."""
    toks = latex_to_text(part).replace("~", " ").split()
    if not 1 <= len(toks) <= 5 or re.search(r"[:\d$]", part):
        return False
    words = 0
    for t in toks:
        if re.fullmatch(r"(?:[A-Z]\.?-?)+\.?", t):
            continue
        if t.lower() in _PARTICLES:
            continue
        if re.fullmatch(r"[A-Z][^\W\d_]*(?:[\'’\-][^\W\d_]+)*\.?", t):
            words += 1
            continue
        return False
    return words >= 1 or len(toks) <= 4


def _names_only(text):
    """True when a text is only a list of person names (commas, 'and', '&')."""
    t = latex_to_text(text).strip(" .,")
    had_etal = bool(re.search(r"\bet\s*al\b", t))
    t = re.sub(r"\bet\s*al\.?", "", t)
    parts = [x.strip(" .") for x in re.split(r",|\band\b|&", t) if x.strip(" .")]
    if not parts:
        return had_etal
    return bool(parts) and all(_looks_like_name(x) for x in parts) and \
        any(re.search(r"[^\W\d_][a-z]", x) for x in parts)


def _split_sentences(text, maxsplit):
    """Split on '. ' that does not end an initial ('J. Smith') or a short abbreviation."""
    return [x for x in re.split(r"(?<![\s.~][A-Z])(?<!\b[A-Z][a-z])\.\s+(?=[A-Z\\{`\"\d])", text,
                                maxsplit=maxsplit) if x.strip()]


def bibitem_to_fields(raw):
    """Free-text reference (body of one \\bibitem) -> BibTeX-like fields dict."""
    fields = {"_raw": re.sub(r"\s+", " ", raw).strip()}
    doi = ""
    for m in re.finditer(r"\\doi\s*\{([^{}]*)\}|doi\.org/(\S+?)[}\s]|\bdoi\s*:?\s*(10\.\S+)", raw, re.I):
        doi = norm_doi(m.group(1) or m.group(2) or m.group(3))
        if doi:
            break
    doi = doi or norm_doi(raw)
    if doi:
        fields["doi"] = doi
    um = re.search(r"\\(?:url|href)\s*\{([^{}]+)\}", raw) or re.search(r"(https?://[^\s{}]+)", raw)
    if um and "doi.org" not in um.group(1):
        fields["url"] = um.group(1).rstrip(".,")
    clean = _strip_ids(raw)
    years = [int(y) for y in _YEAR_RE.findall(latex_to_text(clean))]
    if years:
        paren = re.search(r"\(((?:19|20)\d{2})[a-z]?\)", latex_to_text(clean))
        fields["year"] = paren.group(1) if paren else str(years[-1])

    authors_raw, title_raw, venue_raw = "", "", ""
    # \newblock layout; blocks that are only a stripped link/DOI ("{doi}", ".") do not count
    blocks = [b.strip() for b in re.split(r"\\newblock\b", clean)
              if len(re.sub(r"[^A-Za-z]", "", latex_to_text(b).replace("doi", ""))) >= 4]
    quoted = re.search(r"``(.+?)(?:,|\.)?''|(?<![\\{])\"(.+?)(?:,|\.)?(?<!\\)\"", clean, re.S)
    segs = [x for x in _split_top(clean, r",") if x.strip()]
    colon = re.match(r"^\s*([^:]{2,300}?[^A-Z\s])\s*:\s+(.*)$", latex_to_text(clean), re.S) \
        if len(blocks) < 2 and not quoted else None
    emph_at = next((k for k, x in enumerate(segs) if k > 0 and re.match(
        r"\s*(?:\\emph|\\textit|\{\\em\b|\{\\it\b)", x)), None)
    if len(blocks) >= 2:
        authors_raw = blocks[0]
        title_raw = blocks[1]
        venue_raw = " ".join(blocks[2:])
        if title_raw.lstrip().startswith(("\\emph", "{\\em", "{\\it", "\\textit")):
            fields["_book"] = "1"                 # emphasized 2nd block = book title
    elif quoted:                                  # IEEE: Authors, ``Title,'' Venue
        authors_raw = clean[:quoted.start()]
        title_raw = quoted.group(1) or quoted.group(2)
        venue_raw = clean[quoted.end():]
    elif colon and _names_only(colon.group(1)):   # LNCS: Authors: Title. Venue
        authors_raw = colon.group(1)
        parts = _split_sentences(colon.group(2), 1)
        title_raw, venue_raw = parts[0], (parts[1] if len(parts) > 1 else "")
    elif emph_at is not None and all(_names_only(x) for x in segs[:emph_at]):
        authors_raw = ",".join(segs[:emph_at])    # amsplain: Authors, \emph{Title}, Venue
        title_raw = segs[emph_at]
        venue_raw = ",".join(segs[emph_at + 1:])
    else:
        names = 0
        for x in segs:
            if _names_only(x) or (names and _INITIALS_RE.match(latex_to_text(x).strip() + ".")):
                names += 1
            else:
                break
        if 0 < names < len(segs):                 # elsarticle: A. B, C. D, Title, Venue
            authors_raw = ",".join(segs[:names])
            title_raw = segs[names]
            venue_raw = ",".join(segs[names + 1:])
        else:                                     # Authors. Title. Venue.
            parts = _split_sentences(clean, 2)
            if len(parts) >= 3:
                authors_raw, title_raw, venue_raw = parts[0], parts[1], parts[2]
            elif len(parts) == 2:
                authors_raw, title_raw = parts
            else:
                title_raw = clean
    title = latex_to_text(title_raw).strip(" .,")
    if title and not re.fullmatch(r"[\d\W]*", title):
        fields["title"] = title
    authors = split_free_authors(authors_raw) if authors_raw else []
    if authors:
        fields["author"] = " and ".join(authors)
    _venue_fields(venue_raw, fields)
    aid = find_arxiv_id(fields) or (lambda m: m.group(1) if m else "")(_ARXIV_CTX_RE.search(raw))
    if aid:
        fields["eprint"], fields["archiveprefix"] = aid, "arXiv"
    return fields


def _guess_type(fields):
    if fields.pop("_book", None):
        return "book"
    if fields.get("booktitle"):
        return "inproceedings"
    if fields.get("journal") and not re.search(r"arxiv", fields["journal"], re.I):
        return "article"
    return "misc"


def parse_bibitems(text):
    """All \\bibitem entries of a text (the `thebibliography` blocks of a .tex, or a classic .bbl)."""
    blocks = [m.group(1) for m in re.finditer(
        r"\\begin\s*\{thebibliography\}(.*?)(?:\\end\s*\{thebibliography\}|\Z)", text, re.S)]
    entries = []
    for block in blocks or [text]:
        items = list(_BIBITEM_RE.finditer(block))
        prev_authors = ""
        for k, m in enumerate(items):
            body = block[m.end(): items[k + 1].start() if k + 1 < len(items) else len(block)]
            body = re.split(r"\\end\s*\{thebibliography\}", body)[0]
            key = m.group(2).strip()
            if not key:
                continue
            if prev_authors and re.match(r"\s*\\bysame\b", body):      # amsplain: same authors as before
                body = re.sub(r"\\bysame\b", lambda _: prev_authors, body, count=1)
            fields = bibitem_to_fields(body)
            prev_authors = fields.get("author", "").replace(" and ", ", ")
            entries.append({"key": key, "type": _guess_type(fields), "fields": fields})
    return entries

## 6. biblatex `.bbl` parser

biblatex writes structured records (`\entry{key}{type}{}` … `\endentry`) with `\field{..}{..}`,
`\name{author}{..}` and `\verb{doi}` blocks, so these `.bbl` files convert back to BibTeX exactly
(as suggested in the TA's Q&A on `.bbl` files).

In [ ]:
_NAME_PART = r"(?<![A-Za-z])%s=\{((?:[^{}]|\{[^{}]*\})*)\}"


def _brace_groups(text):
    """Contents of the top-level `{...}` groups of a text, in order."""
    out, i = [], text.find("{")
    while i >= 0:
        close = _matching_brace(text, i)
        if close < 0:
            break
        out.append(text[i + 1:close])
        i = text.find("{", close + 1)
    return out


def parse_biblatex_bbl(text):
    entries = []
    for m in re.finditer(r"\\entry\s*\{([^{}]+)\}\s*\{([^{}]+)\}\s*\{[^{}]*\}(.*?)\\endentry", text, re.S):
        key, etype, body = m.group(1).strip(), m.group(2).strip().lower(), m.group(3)
        fields = {}
        for fm in re.finditer(r"\\field\s*\{(\w+)\}\s*\{", body):
            close = _matching_brace(body, fm.end() - 1)
            if close > 0:
                fields[fm.group(1).lower()] = body[fm.end():close]
        for vm in re.finditer(r"\\verb\s*\{(\w+)\}\s*\\verb\s+(.*?)\s*\\endverb", body, re.S):
            fields[vm.group(1).lower()] = re.sub(r"\s+", "", vm.group(2))
        for nm in re.finditer(r"\\name\s*\{(author|editor)\}\s*\{\d+\}\s*\{[^{}]*\}\s*\{", body):
            close = _matching_brace(body, nm.end() - 1)
            if close < 0:
                continue
            block, names = body[nm.end():close], []
            for person in _brace_groups(block):       # one `{{hash=..}{family={..},given={..}}}` per person
                fam = re.search(_NAME_PART % "family", person)
                giv = re.search(_NAME_PART % "given", person)
                if fam:
                    fam, giv = fam.group(1), (giv.group(1) if giv else "")
                else:                                 # biblatex < 3: {{hash}{Last}{L.}{First}{F.}{}{}{}{}}
                    parts = _brace_groups(person)
                    fam, giv = (parts[1] if len(parts) > 1 else ""), (parts[3] if len(parts) > 3 else "")
                if fam:
                    names.append(f"{fam}, {giv}" if giv else fam)
            if names:
                fields[nm.group(1)] = " and ".join(names)
        if "journaltitle" in fields:
            fields.setdefault("journal", fields.pop("journaltitle"))
        if "year" not in fields and fields.get("date"):
            fields["year"] = str(parse_year(fields["date"]) or "")
        if fields.get("eprinttype"):
            fields.setdefault("archiveprefix", fields["eprinttype"])
        entries.append({"key": key, "type": etype, "fields": fields})
    return entries


def parse_bbl(text):
    """A .bbl file of either kind (biblatex records or classic \\bibitem list)."""
    if re.search(r"\\entry\s*\{", text) and "\\endentry" in text:
        return parse_biblatex_bbl(text)
    return parse_bibitems(text)

## 7. Cite commands: find keys and rename them

One scanner serves both jobs. It matches every command whose name contains `cite`
(`\cite`, `\citep`, `\citet`, `\citeauthor`, `\citeyear`, `\citealp`, `\Citep`, `\parencite`,
`\textcite`, `\autocite`, ...) plus `\nocite`, with a star, up to two optional arguments
(`\citep[see][p.~5]{a,b}`) and the biblatex multi-cite form (`\cites[..]{a}[..]{b}`).
Keys repeated after renaming (`\cite{a,b}` with `b → a`) are kept once, keys that are not in the
mapping stay as they are, and a command without any renamed key keeps its exact text.

In [ ]:
_CITE_CMD_RE = re.compile(r"\\([A-Za-z]*[cC]ite[A-Za-z]*|nocite)(?![A-Za-z])(\*?)")
_NOT_CITE = {"citestyle", "citeindextrue", "citeindexfalse", "citeauthorstyle", "bibcite", "bstctlcite"}


def _scan_cites(text):
    """Yield (start, end, [(key_start, key_end), ...]) for each cite command."""
    n = len(text)
    for m in _CITE_CMD_RE.finditer(text):
        name = m.group(1)
        if name in _NOT_CITE or name.startswith("citation"):
            continue
        i, groups, multi = m.end(), [], name.endswith("cites")
        while True:
            j = i
            while j < n and text[j] in " \t\n":
                j += 1
            while j < n and text[j] == "[":       # optional arguments
                close = text.find("]", j)
                if close < 0:
                    break
                j = close + 1
                while j < n and text[j] in " \t\n":
                    j += 1
            if j < n and text[j] == "{":
                close = _matching_brace(text, j)
                if close < 0:
                    break
                groups.append((j + 1, close))
                i = close + 1
                if multi:
                    continue
            break
        if groups:
            yield m.start(), i, groups


def cited_keys(text):
    """Counter of cited keys in a text (\\nocite{*} counts as key '*')."""
    keys = Counter()
    for _, _, groups in _scan_cites(text or ""):
        for a, b in groups:
            for k in text[a:b].split(","):
                k = k.strip()
                if k and "#" not in k and "\\" not in k:    # skip macro parameters (#1)
                    keys[k] += 1
    return keys


def rename_cites(text, mapping):
    """Rewrite cite keys with `mapping` {old_key: new_key}; unknown keys stay unchanged.

    Only the key lists that contain a mapped key are rewritten, so every other command keeps
    its exact text. Two old keys that map to the same new key are written once.
    """
    if not mapping or not text or "cite" not in text:
        return text
    spans = sorted({g for _, _, groups in _scan_cites(text) for g in groups})
    out, pos = [], 0
    for a, b in spans:
        raw = text[a:b]
        if a < pos or "\\" in raw or "{" in raw:      # not a plain key list (e.g. a macro definition)
            continue
        keys = [k.strip() for k in raw.split(",")]
        if not any(mapping.get(k, k) != k for k in keys):
            continue
        new = list(dict.fromkeys(mapping.get(k, k) for k in keys if k))
        out.append(text[pos:a])
        out.append(",".join(new))
        pos = b
    out.append(text[pos:])
    return "".join(out)

## 8. References of one version and their source label

Input: the version folder (`DATA_DIR/<yymm-id>/vN/`) and the **gathered** LaTeX of that version
(output of Member 2's `gather()`, comments already removed). Without `tex`, the `.tex` files of
the folder are concatenated with comments removed (good enough for declarations and cite keys).

Source label, in this order:

| label | meaning |
|---|---|
| `bib` | entries read from the `.bib` files named by `\bibliography{..}` / `\addbibresource{..}` |
| `bibitem` | `\bibitem` list written directly in the `.tex` |
| `bbl` | no usable `.bib`, entries recovered from a `.bbl` (if Member 2 keeps `.bbl`) |
| `missing_bbl` | a bibliography is declared but neither its `.bib` nor a `.bbl` is in the source |
| `none` | no bibliography at all (or an empty-source version) |

A `.bib` and `\bibitem`s found together are both kept (union). If the declared name does not
match any file but the folder holds `.bib` files, those are used (`bib_fallback = True`).

In [ ]:
_DECL_RE = re.compile(r"\\(?:bibliography|addbibresource|addglobalbib|addsectionbib)\s*(?:\[[^\]]*\])?\s*\{([^{}]*)\}")


def _strip_comments(tex):
    return re.sub(r"(?<!\\)%[^\n]*", "", tex)


def _fallback_tex(version_dir):
    """Concatenate all .tex files of a version (used only when no gathered text is given)."""
    texts = []
    for p in sorted(Path(version_dir).rglob("*")):
        if p.is_file() and p.suffix.lower() == ".tex":
            try:
                texts.append(_strip_comments(p.read_text(encoding="utf-8", errors="replace")))
            except OSError:
                pass
    return "\n".join(texts)


def _read_text(path):
    raw = Path(path).read_bytes()
    for enc in ("utf-8", "latin-1"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


@dataclass
class VersionRefs:
    source: str                      # one of SOURCE_LABELS
    entries: list                    # [{'key','type','fields'}] after the cited-only filter
    n_raw: int = 0                   # entries read before the filter
    declared: list = field(default_factory=list)   # names in \bibliography / \addbibresource
    bib_files: list = field(default_factory=list)  # files actually read (relative paths)
    missing: list = field(default_factory=list)    # declared names with no file
    bib_fallback: bool = False
    cite_keys: Counter = field(default_factory=Counter)
    uncited_dropped: int = 0
    unresolved_cites: int = 0        # cited keys with no entry (often from a missing .bbl)


def extract_version_refs(version_dir, tex=None, only_cited=None):
    """Read every reference entry of one version and label where they came from."""
    only_cited = BIB_ONLY_CITED if only_cited is None else only_cited
    vdir = Path(version_dir)
    tex = _fallback_tex(vdir) if tex is None else tex
    files = [p for p in vdir.rglob("*") if p.is_file()] if vdir.is_dir() else []
    bibs = [p for p in files if p.suffix.lower() == ".bib"]
    bbls = [p for p in files if p.suffix.lower() == ".bbl"]

    declared = []
    for m in _DECL_RE.finditer(tex):
        declared += [d.strip() for d in m.group(1).split(",") if d.strip()]
    declared = list(dict.fromkeys(declared))

    def rel(p):
        return p.relative_to(vdir).as_posix().lower()

    by_rel = {re.sub(r"\.bib$", "", rel(p)): p for p in bibs}
    by_base = defaultdict(list)
    for p in bibs:
        by_base[p.stem.lower()].append(p)
    chosen, missing = [], []
    for name in declared:
        key = re.sub(r"\.bib$", "", name.lower().lstrip("./"))
        hit = by_rel.get(key) or (by_base.get(Path(key).name) or [None])[0]
        if hit is None:   # paths relative to a sub-folder that holds the main .tex
            hit = next((p for r, p in by_rel.items() if r.endswith("/" + key)), None)
        (chosen if hit is not None else missing).append(hit if hit is not None else name)
    chosen = list(dict.fromkeys(chosen))
    fallback = False
    if declared and not chosen and bibs:
        chosen, fallback = sorted(bibs), True

    entries = []
    for p in chosen:
        entries += parse_bib(_read_text(p))
    bib_entries = len(entries)
    item_entries = parse_bibitems(tex) if "\\bibitem" in tex else []
    entries += item_entries
    bbl_entries = []
    if not entries and bbls:
        # prefer the .bbl named like a .tex file (main.tex -> main.bbl), else the largest one
        tex_stems = {p.stem.lower() for p in files if p.suffix.lower() == ".tex"}
        bbls.sort(key=lambda p: (p.stem.lower() not in tex_stems, -p.stat().st_size))
        bbl_entries = parse_bbl(_read_text(bbls[0]))
        entries = bbl_entries

    if bib_entries:
        source = "bib"
    elif item_entries:
        source = "bibitem"
    elif bbl_entries:
        source = "bbl"
    elif declared:
        source = "missing_bbl"
    else:
        source = "none"

    cites = cited_keys(tex)
    n_raw = len(entries)
    dropped = 0
    if only_cited and source == "bib" and cites and "*" not in cites:
        from_items = {id(e) for e in item_entries}             # \bibitem entries are always kept
        keep = [e for e in entries if e["key"] in cites or id(e) in from_items]
        dropped = n_raw - len(keep)
        entries = keep
    have = {e["key"] for e in entries}
    unresolved = sum(1 for k in cites if k != "*" and k not in have)
    return VersionRefs(source=source, entries=entries, n_raw=n_raw, declared=declared,
                       bib_files=[p.relative_to(vdir).as_posix() for p in chosen], missing=missing,
                       bib_fallback=fallback, cite_keys=cites, uncited_dropped=dropped,
                       unresolved_cites=unresolved)

## 9. Reference deduplication across versions

Every entry of every version is a node; nodes that refer to the same work are merged with a
union–find. Two nodes are merged when **one** rule holds:

1. same key inside one version (duplicate entry in the `.bib`);
2. same **DOI**;
3. same **arXiv id** (from `eprint`, URL, `journal = {arXiv preprint ...}`, arXiv DOI);
4. same **normalized title** and compatible years (equal, or one is missing). Titles with fewer
   than 4 words also need the same first-author surname;
5. same **key in two versions**, **both** entries have a title, and the titles share at least 60%
   of their words. Purely numeric keys (`15`) never use this rule, because numbered reference
   lists are renumbered between versions.

Rules 2–4 are the statement's "same underlying reference". Rule 5 is **our own choice**: it catches
a title that was edited between versions (typo fixed, subtitle added) while the key stayed the same.

A merge is refused when the two groups carry different non-arXiv DOIs or different arXiv ids,
so two distinct works with the same generic title stay apart.

For each group: the **chosen key** is the key used in the **newest version** (most cited there
when a version has two duplicate keys); the **fields are the union** of all members, newer values
winning on conflicts. `key_map[v]` lists the keys of version `v` that must be renamed.
Every merge is written to `merge_log` (the rule and the two `(version, key)` pairs), so each rule
can be audited on real data.

In [ ]:
DEDUP_RULES = ("same_key_in_version", "doi", "arxiv_id", "title_year", "same_key_across_versions")
KEY_TITLE_OVERLAP = 0.6      # rule 5: minimum word overlap (Jaccard) between the two titles


def _prepare(entry):
    """Cached matching features of one entry."""
    f = entry["fields"]
    title = latex_to_text(f.get("title", ""))
    authors = split_bib_authors(f.get("author", "")) if f.get("author") else []
    doi = norm_doi(f.get("doi", ""))
    arx = find_arxiv_id(f)
    if doi.startswith("10.48550/arxiv."):
        doi = ""
    return {"ntitle": norm_title(title), "year": parse_year(f.get("year") or f.get("date")),
            "doi": doi, "arxiv": arx.lower(), "surname": surname_key(authors[0]) if authors else ""}


def _title_overlap(a, b):
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / max(1, len(sa | sb))


class _UnionFind:
    def __init__(self, feats):
        self.parent = list(range(len(feats)))
        self.dois = [{f["doi"]} - {""} for f in feats]
        self.arx = [{f["arxiv"]} - {""} for f in feats]
        self.years = [{f["year"]} - {None} for f in feats]

    def find(self, i):
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i

    def union(self, i, j):
        a, b = self.find(i), self.find(j)
        if a == b:
            return True
        if (self.dois[a] and self.dois[b] and not self.dois[a] & self.dois[b]) or \
                (self.arx[a] and self.arx[b] and not self.arx[a] & self.arx[b]):
            return False                       # conflicting identifiers: different works
        self.parent[b] = a
        self.dois[a] |= self.dois[b]
        self.arx[a] |= self.arx[b]
        self.years[a] |= self.years[b]
        return True


@dataclass
class DedupResult:
    entries: list                    # unified entries: {'key','type','fields','versions','merged_keys'}
    key_map: dict                    # {version: {old_key: chosen_key}} (only keys that change)
    n_before: int = 0                # entries summed over versions (after cited-only filter)
    n_after: int = 0
    merges_by_rule: Counter = field(default_factory=Counter)
    merge_log: list = field(default_factory=list)   # one row per merge: rule + the two (version, key)


def dedup_references(version_entries, cite_counts=None):
    """{version: [entries]} -> DedupResult. `cite_counts` = {version: Counter} breaks key ties."""
    nodes = [(int(v), e) for v in sorted(version_entries, key=int) for e in version_entries[v]]
    if not nodes:
        return DedupResult(entries=[], key_map={})
    feats = [_prepare(e) for _, e in nodes]
    uf = _UnionFind(feats)
    rules, log = Counter(), []

    def link(rule, i, j):
        if uf.find(i) != uf.find(j) and uf.union(i, j):
            rules[rule] += 1
            log.append({"rule": rule, "version_a": nodes[i][0], "key_a": nodes[i][1]["key"],
                        "version_b": nodes[j][0], "key_b": nodes[j][1]["key"]})

    def merge_by(rule, keyfunc):
        groups = defaultdict(list)
        for i in range(len(nodes)):
            k = keyfunc(i)
            if k:
                groups[k].append(i)
        for members in groups.values():
            for j in members[1:]:
                link(rule, members[0], j)

    merge_by("same_key_in_version", lambda i: (nodes[i][0], nodes[i][1]["key"]))
    merge_by("doi", lambda i: feats[i]["doi"])
    merge_by("arxiv_id", lambda i: feats[i]["arxiv"])

    # rule 4: same normalized title, years compatible (compared on the groups built so far, so
    # "2001" and "2005" are not joined through a third entry that has no year)
    by_title = defaultdict(list)
    for i, f in enumerate(feats):
        if f["ntitle"] and len(f["ntitle"]) >= 10:
            by_title[f["ntitle"]].append(i)
    for members in by_title.values():
        for x in range(len(members)):
            for y in range(x + 1, len(members)):
                i, j = members[x], members[y]
                a, b = uf.find(i), uf.find(j)
                if a == b:
                    continue
                if uf.years[a] and uf.years[b] and not uf.years[a] & uf.years[b]:
                    continue
                fi, fj = feats[i], feats[j]
                if len(fi["ntitle"].split()) < 4 and fi["surname"] and fj["surname"] and \
                        fi["surname"] != fj["surname"]:
                    continue
                link("title_year", i, j)

    # rule 5 (our own choice, stricter than "same key"): the key is reused in another version AND
    # both entries have a title AND the titles share >= 60% of their words. Purely numeric keys
    # ("15") are skipped: numbered reference lists are renumbered between versions.
    by_key = defaultdict(list)
    for i, (v, e) in enumerate(nodes):
        by_key[e["key"]].append(i)
    for key, members in by_key.items():
        if re.fullmatch(r"\d+", key):
            continue
        for x in range(len(members)):
            for y in range(x + 1, len(members)):
                i, j = members[x], members[y]
                if nodes[i][0] == nodes[j][0]:
                    continue
                ti, tj = feats[i]["ntitle"], feats[j]["ntitle"]
                if ti and tj and _title_overlap(ti, tj) >= KEY_TITLE_OVERLAP:
                    link("same_key_across_versions", i, j)

    clusters = defaultdict(list)
    for i in range(len(nodes)):
        clusters[uf.find(i)].append(i)

    cite_counts = cite_counts or {}
    unified, key_map, used = [], defaultdict(dict), set()
    # newest clusters first so that, on a key clash, the newest version keeps the plain key
    order = sorted(clusters.values(), key=lambda m: (-max(nodes[i][0] for i in m), min(m)))
    for members in order:
        newest = max(nodes[i][0] for i in members)
        cands = [nodes[i][1]["key"] for i in members if nodes[i][0] == newest]
        counts = cite_counts.get(newest, Counter())
        chosen = max(dict.fromkeys(cands), key=lambda k: (counts.get(k, 0), -cands.index(k)))
        base, n = chosen, 2
        while chosen in used:                   # same key reused for a different work
            chosen, n = f"{base}_{n}", n + 1
        used.add(chosen)
        merged, etype, versions = {}, "misc", set()
        for i in sorted(members, key=lambda i: nodes[i][0]):     # oldest -> newest: newer wins
            v, e = nodes[i]
            versions.add(v)
            for k, val in e["fields"].items():
                if val not in ("", None):
                    merged[k] = val
            if e["type"] != "misc" or etype == "misc":
                etype = e["type"]
            if e["key"] != chosen:
                key_map[v][e["key"]] = chosen
        unified.append({"key": chosen, "type": etype, "fields": merged, "versions": sorted(versions),
                        "merged_keys": sorted({nodes[i][1]["key"] for i in members})})
    unified.sort(key=lambda e: e["key"].lower())
    return DedupResult(entries=unified, key_map={v: dict(m) for v, m in key_map.items()},
                       n_before=len(nodes), n_after=len(unified), merges_by_rule=rules, merge_log=log)

## 10. Output: `bibtex` table rows, `key_map` rows, BibTeX text

`bibtex` rows follow the statement exactly: `arxiv_id, citation_key, entry_type, title,
authors (list), year (int or None), doi, journal, url`. `journal` falls back to `booktitle`
(the venue of conference papers) and `url` falls back to the arXiv abstract page when only an
arXiv id is known, so Milestone 3 keeps this information.

In [ ]:
def entry_row(arxiv_id, entry):
    f = entry["fields"]
    aid = find_arxiv_id(f)
    url = f.get("url", "") or (f"https://arxiv.org/abs/{aid}" if aid else "")
    return {
        "arxiv_id": arxiv_id,
        "citation_key": entry["key"],
        "entry_type": entry["type"],
        "title": latex_to_text(f.get("title", "")) or None,
        "authors": split_bib_authors(f.get("author", "")),
        "year": parse_year(f.get("year") or f.get("date")),
        "doi": norm_doi(f.get("doi", "")) or None,
        "journal": latex_to_text(f.get("journal") or f.get("booktitle") or "") or None,
        "url": re.sub(r"\\_", "_", url).strip() or None,
    }


def bibtex_rows(arxiv_id, entries):
    return [entry_row(arxiv_id, e) for e in entries]


def key_map_rows(arxiv_id, key_map):
    return [{"arxiv_id": arxiv_id, "version": int(v), "old_key": old, "new_key": new}
            for v, m in sorted(key_map.items()) for old, new in sorted(m.items())]


def entry_to_bibtex(entry):
    """Unified entry -> BibTeX text (private fields starting with '_' are left out)."""
    lines = [f"@{entry['type']}{{{entry['key']},"]
    for k, v in entry["fields"].items():
        if k.startswith("_") or v in ("", None):
            continue
        v = str(v).replace("\n", " ")
        if v.count("{") != v.count("}"):
            v = v.replace("{", "").replace("}", "")
        lines.append(f"  {k} = {{{v}}},")
    lines.append("}")
    return "\n".join(lines)

## 11. One paper end to end

`process_paper_bib(arxiv_id, versions)` is what `parse_paper()` (Member 1) calls after building
the trees. `versions` maps the version number to `(version_dir, gathered_tex)`; an empty-source
version passes `(dir, "")` and is labelled `none`. It returns the `bibtex` rows, the `key_map`
(apply it with `rename_cites` to every element of that version **before** full-text dedup),
the per-version stats, the per-paper stats, the unified entries (`entry_to_bibtex` writes a `.bib`
file) and one row per merge (`merge_rows`).

In [ ]:
def process_paper_bib(arxiv_id, versions, only_cited=None):
    t0 = time.perf_counter()
    refs = {int(v): extract_version_refs(d, tex, only_cited) for v, (d, tex) in versions.items()}
    result = dedup_references({v: r.entries for v, r in refs.items()},
                              cite_counts={v: r.cite_keys for v, r in refs.items()})
    version_stats = [{
        "arxiv_id": arxiv_id, "version": v, "bib_source": r.source, "n_raw": r.n_raw,
        "n_entries": len(r.entries), "uncited_dropped": r.uncited_dropped,
        "n_cite_keys": len([k for k in r.cite_keys if k != "*"]), "unresolved_cites": r.unresolved_cites,
        "declared": ";".join(r.declared), "missing": ";".join(map(str, r.missing)),
        "bib_fallback": r.bib_fallback,
    } for v, r in sorted(refs.items())]
    paper_stats = {"arxiv_id": arxiv_id, "versions": len(refs), "entries_before": result.n_before,
                   "entries_after": result.n_after, "renamed_keys": sum(len(m) for m in result.key_map.values()),
                   **{f"rule_{k}": result.merges_by_rule.get(k, 0) for k in DEDUP_RULES},
                   "seconds": round(time.perf_counter() - t0, 4)}
    return {"rows": bibtex_rows(arxiv_id, result.entries), "key_map": result.key_map,
            "key_map_rows": key_map_rows(arxiv_id, result.key_map), "version_stats": version_stats,
            "paper_stats": paper_stats, "entries": result.entries,
            "merge_rows": [{"arxiv_id": arxiv_id, **m} for m in result.merge_log]}

## 12. Standalone run over the crawled corpus

Lets Member 3 build and check the `bibtex` table on its own, before `15_run_parsing` exists.
It uses Member 2's `gather()` when that function is loaded, otherwise the fallback text.
A paper that raises is written to `bib_errors` and the run continues.

Output, one file per table (Parquet when `pandas` and `pyarrow` are installed, else JSON lines):

| Table | One row per | Columns |
|---|---|---|
| `bibtex` | unified reference | the 9 columns of the statement |
| `bib_key_map` | renamed key | `arxiv_id, version, old_key, new_key` |
| `bib_version_stats` | version | source label, entries read / kept, cited keys, cited keys without an entry |
| `bib_paper_stats` | paper | entries before / after deduplication, renamed keys, merges per rule, seconds |
| `bib_merges` | merge | `arxiv_id, rule, version_a, key_a, version_b, key_b` |
| `bib_errors` | failed paper | `arxiv_id, error` |

In [ ]:
# Columns of the side tables (fixed, so an empty table still has a schema)
BIB_TABLES = {
    "bibtex": BIBTEX_COLUMNS,
    "bib_key_map": ["arxiv_id", "version", "old_key", "new_key"],
    "bib_version_stats": ["arxiv_id", "version", "bib_source", "n_raw", "n_entries", "uncited_dropped",
                          "n_cite_keys", "unresolved_cites", "declared", "missing", "bib_fallback"],
    "bib_paper_stats": ["arxiv_id", "versions", "entries_before", "entries_after", "renamed_keys",
                        *[f"rule_{r}" for r in DEDUP_RULES], "seconds"],
    "bib_merges": ["arxiv_id", "rule", "version_a", "key_a", "version_b", "key_b"],
    "bib_errors": ["arxiv_id", "error"],
}


def _version_dirs(paper_dir):
    out = {}
    for p in Path(paper_dir).iterdir():
        m = re.fullmatch(r"v(\d+)", p.name)
        if p.is_dir() and m:
            out[int(m.group(1))] = p
    return out


def _gathered(vdir):
    g = globals().get("gather")
    if callable(g):
        try:
            res = g(vdir)
            return res if isinstance(res, str) else (res or {}).get("text")
        except Exception:
            return None
    return None


def save_table(rows, path, columns=None):
    """Write rows as Parquet (pandas + pyarrow) or JSON lines; returns the path written."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import pandas as pd
        import pyarrow  # noqa: F401
    except ImportError:
        out = path.with_suffix(".jsonl")
        with open(out, "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        return out
    df = pd.DataFrame(rows, columns=columns)
    if "year" in df:
        df["year"] = df["year"].astype("Int64")       # nullable integer, as in the statement
    out = path.with_suffix(".parquet")
    df.to_parquet(out, index=False)
    return out


def build_bibtex_tables(data_dir, out_dir, ids=None, limit=None, progress_every=500):
    """Run process_paper_bib on every paper folder of `data_dir`; save the tables of BIB_TABLES.

    Returns (written paths, tables as lists of rows, errors). A paper that raises is recorded in
    `bib_errors` and the run goes on.
    """
    papers = sorted(p for p in Path(data_dir).iterdir() if p.is_dir())
    if ids:
        wanted = {i.replace(".", "-").replace("/", "_") for i in ids}
        papers = [p for p in papers if p.name in wanted]
    papers = papers[:limit] if limit else papers
    tables = {name: [] for name in BIB_TABLES}
    errors, t0 = tables["bib_errors"], time.perf_counter()
    for k, pdir in enumerate(papers, 1):
        # folder name -> arXiv id: '2310-12345' -> '2310.12345', 'hep-th_9901001' -> 'hep-th/9901001'
        arxiv_id = (pdir.name.replace("-", ".", 1) if re.fullmatch(r"\d{4}-\d{4,5}", pdir.name)
                    else pdir.name.replace("_", "/"))
        try:
            versions = {v: (d, _gathered(d)) for v, d in _version_dirs(pdir).items()}
            res = process_paper_bib(arxiv_id, versions)
            tables["bibtex"] += res["rows"]
            tables["bib_key_map"] += res["key_map_rows"]
            tables["bib_version_stats"] += res["version_stats"]
            tables["bib_paper_stats"].append(res["paper_stats"])
            tables["bib_merges"] += res["merge_rows"]
        except Exception as e:                      # one bad paper must not stop the run
            errors.append({"arxiv_id": arxiv_id, "error": f"{type(e).__name__}: {e}"})
        if progress_every and k % progress_every == 0:
            print(f"[bibtex] {k}/{len(papers)} papers, {time.perf_counter() - t0:.1f}s", flush=True)
    written = {name: save_table(rows, Path(out_dir) / name, BIB_TABLES[name]) for name, rows in tables.items()}
    print(f"[bibtex] {len(papers)} papers, {len(tables['bibtex'])} entries, {len(errors)} errors, "
          f"{time.perf_counter() - t0:.1f}s")
    return written, tables, errors


def bibtex_summary(tables, errors, seconds=None):
    """Print (and return) the numbers reported in the book for one run of build_bibtex_tables."""
    labels = Counter(r["bib_source"] for r in tables["bib_version_stats"])
    summary = {
        "papers": len(tables["bib_paper_stats"]),
        "versions": len(tables["bib_version_stats"]),
        "versions_by_source": {label: labels.get(label, 0) for label in SOURCE_LABELS},
        "entries_before": sum(p["entries_before"] for p in tables["bib_paper_stats"]),
        "entries_after": sum(p["entries_after"] for p in tables["bib_paper_stats"]),
        "renamed_keys": len(tables["bib_key_map"]),
        "merges_by_rule": {r: sum(p[f"rule_{r}"] for p in tables["bib_paper_stats"]) for r in DEDUP_RULES},
        "errors": len(errors),
        "seconds": None if seconds is None else round(seconds, 1),
    }
    for name, value in summary.items():
        print(f"{name:>20}: {value}")
    for e in errors[:10]:
        print(f"{'error':>20}: {e['arxiv_id']}  {e['error']}")
    return summary

## 13. Tests (run only when this notebook is opened on its own: `RUN_TEST = True`)

Offline unit tests for every rule above, including the statement's own example
(`vaswani2017` → `vit_transformer`, fields unioned, `year` dropped in v3).

In [ ]:
import contextlib
import io
import tempfile
import unittest


class BibtexTests(unittest.TestCase):
    # ---------- text helpers ----------
    def test_latex_to_text_accents_and_fonts(self):
        self.assertEqual(latex_to_text(r'Lambert M{\"u}nster'), "Lambert Münster")
        self.assertEqual(latex_to_text(r"{\em Physical Review E}"), "Physical Review E")
        self.assertEqual(latex_to_text(r"Meli\`a-Segu\'\i"), "Melià-Seguí")
        self.assertEqual(latex_to_text(r"Lincoln~F. Chayes"), "Lincoln F. Chayes")
        self.assertEqual(latex_to_text(r"\emph{Deep {L}earning} \& {AI}"), "Deep Learning & AI")

    def test_norm_title_ignores_case_braces_accents(self):
        self.assertEqual(norm_title(r"{Attention} is All you {N}eed"), norm_title("attention is all you need."))
        self.assertEqual(norm_title(r"Caf\'e"), "cafe")

    def test_ids(self):
        self.assertEqual(norm_doi("https://doi.org/10.1103/PhysRevE.58.2749."), "10.1103/physreve.58.2749")
        self.assertEqual(find_arxiv_id({"journal": "arXiv preprint arXiv:1706.03762"}), "1706.03762")
        self.assertEqual(find_arxiv_id({"eprint": "1706.03762v5", "archiveprefix": "arXiv"}), "1706.03762")
        self.assertEqual(find_arxiv_id({"url": "https://arxiv.org/abs/hep-th/9901001v2"}), "hep-th/9901001")
        self.assertEqual(find_arxiv_id({"doi": "10.48550/arXiv.2101.00001"}), "2101.00001")
        self.assertEqual(find_arxiv_id({"journal": "Phys. Rev. 1706.03762"}), "")

    def test_authors(self):
        self.assertEqual(split_bib_authors("Vaswani, Ashish and Shazeer, Noam and others"),
                         ["Ashish Vaswani", "Noam Shazeer"])
        self.assertEqual(split_bib_authors(r"{Article 29 Working Party}"), ["Article 29 Working Party"])
        self.assertEqual(split_free_authors("An, X., Zhu, X., and Fu, Y."), ["X. An", "X. Zhu", "Y. Fu"])
        self.assertEqual(split_free_authors(r"H.\ Cohn, A.\ Kumar"), ["H. Cohn", "A. Kumar"])

    # ---------- .bib ----------
    def test_parse_bib_values_strings_concat(self):
        src = r"""
        @string{prl = "Phys. Rev. Lett."}
        @comment{ignored @article{x, title={no}} }
        @Article{a1, title = {A {GPU} study}, author = "Doe, John and Roe, R.",
          journal = prl # " (short)", year = 2020, month = mar, doi = {10.1/ABC}}
        @inproceedings(b2, title="Paren form", booktitle={NeurIPS}, year={2021})
        @misc{broken, title = {unbalanced
        @book{c3, title={After broken}, year={1999}}
        """
        es = {e["key"]: e for e in parse_bib(src)}
        self.assertEqual(es["a1"]["type"], "article")
        self.assertEqual(es["a1"]["fields"]["journal"], "Phys. Rev. Lett. (short)")
        self.assertEqual(es["a1"]["fields"]["month"], "3")
        self.assertEqual(es["a1"]["fields"]["year"], "2020")
        self.assertEqual(es["b2"]["fields"]["booktitle"], "NeurIPS")
        self.assertIn("c3", es)
        self.assertNotIn("x", es)

    # ---------- \bibitem ----------
    def test_bibitem_newblock_style(self):
        e = parse_bibitems(r"""\begin{thebibliography}{10}
\bibitem{MW2023}
Lambert M{\"u}nster and Martin Weigel.
\newblock Cluster percolation in the two-dimensional {Ising} spin glass.
\newblock {\em Physical Review E}, 107(5):054103, 2023.
\bibitem[An et~al.(2021)]{9607608}
An, X., Zhu, X., and Fu, Y.
\newblock {Partial FC}: Training 10 million identities on a single machine.
\newblock In \emph{2021 IEEE/CVF ICCV Workshops}, pp.\  1445--1449, 2021.
\newblock \doi{10.1109/ICCVW54120.2021.00166}.
\end{thebibliography}""")
        a, b = e
        self.assertEqual(a["key"], "MW2023")
        self.assertEqual(a["fields"]["title"], "Cluster percolation in the two-dimensional Ising spin glass")
        self.assertEqual(a["fields"]["journal"], "Physical Review E")
        self.assertEqual(a["fields"]["year"], "2023")
        self.assertEqual(split_bib_authors(a["fields"]["author"]), ["Lambert Münster", "Martin Weigel"])
        self.assertEqual(a["type"], "article")
        self.assertEqual(b["key"], "9607608")
        self.assertEqual(b["fields"]["doi"], "10.1109/iccvw54120.2021.00166")
        self.assertEqual(b["type"], "inproceedings")
        self.assertEqual(b["fields"]["pages"], "1445--1449")

    def test_bibitem_ieee_quoted_title(self):
        (e,) = parse_bibitems(r"""\bibitem{Adame2014}
T.~Adame, A.~Bel, and R.~Pous, ``IEEE 802.11ah: The WiFi Approach for M2M Communications,''
\emph{IEEE Wireless Communications}, vol.~21, no.~6, pp.~144--152, 2014.""")
        f = e["fields"]
        self.assertEqual(f["title"], "IEEE 802.11ah: The WiFi Approach for M2M Communications")
        self.assertEqual(f["journal"], "IEEE Wireless Communications")
        self.assertEqual(f["year"], "2014")
        self.assertEqual(f["pages"], "144--152")
        self.assertEqual(split_bib_authors(f["author"]), ["T. Adame", "A. Bel", "R. Pous"])

    def test_bibitem_comma_style(self):
        (e,) = parse_bibitems(r"""\bibitem{CohnK:07}
H.\ Cohn, A.\ Kumar,
Universally optimal distribution of points on spheres,
J.\ Amer.\ Math.\ Soc.\ 20 (2007) 99--148.""")
        f = e["fields"]
        self.assertEqual(f["title"], "Universally optimal distribution of points on spheres")
        self.assertEqual(f["year"], "2007")
        self.assertTrue(f["journal"].startswith("J. Amer. Math. Soc"))
        self.assertEqual(split_bib_authors(f["author"]), ["H. Cohn", "A. Kumar"])

    def test_bibitem_arxiv_and_href_doi(self):
        (e,) = parse_bibitems(r"""\bibitem{x} A.~B. Smith, Some title here, arXiv:2101.00001 (2021).
\newblock \href {https://doi.org/10.1021/i160023a012} {doi}""")
        self.assertEqual(e["fields"]["eprint"], "2101.00001")
        self.assertEqual(e["fields"]["doi"], "10.1021/i160023a012")

    # ---------- biblatex .bbl ----------
    def test_biblatex_bbl(self):
        bbl = r"""\entry{vaswani2017}{inproceedings}{}
  \name{author}{2}{}{%
    {{hash=1}{family={Vaswani},familyi={V\bibinitperiod},given={Ashish},giveni={A\bibinitperiod}}}%
    {{hash=2}{family={Shazeer},familyi={S\bibinitperiod},given={Noam},giveni={N\bibinitperiod}}}%
  }
  \field{title}{Attention is All you Need}
  \field{booktitle}{NeurIPS}
  \field{year}{2017}
  \verb{doi}
  \verb 10.5555/3295222
  \endverb
\endentry"""
        (e,) = parse_bbl(bbl)
        self.assertEqual(e["key"], "vaswani2017")
        self.assertEqual(split_bib_authors(e["fields"]["author"]), ["Ashish Vaswani", "Noam Shazeer"])
        self.assertEqual(e["fields"]["doi"], "10.5555/3295222")
        self.assertEqual(entry_row("x", e)["journal"], "NeurIPS")

    # ---------- cite renaming ----------
    def test_cited_keys_and_rename(self):
        t = (r"See \cite{a, b} and \citep[see][p.~5]{b,c}, \citet*{a}, \citeauthor{d}, "
             r"\nocite{e} \parencite{a} \cites[1]{a}[2]{b} \citestyle{a} \Citep{b}.")
        keys = cited_keys(t)
        self.assertEqual(keys["a"], 4)
        self.assertNotIn("citestyle", keys)
        out = rename_cites(t, {"b": "a", "e": "E"})
        self.assertIn(r"\cite{a}", out)                  # a,b -> a (kept once)
        self.assertIn(r"\citep[see][p.~5]{a,c}", out)
        self.assertIn(r"\nocite{E}", out)
        self.assertIn(r"\cites[1]{a}[2]{a}", out)
        self.assertIn(r"\Citep{a}", out)
        self.assertIn(r"\citestyle{a}", out)
        self.assertEqual(rename_cites(t, {}), t)

    # ---------- dedup ----------
    def _e(self, key, **fields):
        return {"key": key, "type": "article", "fields": fields}

    def test_statement_example(self):
        v1 = [self._e("vaswani2017", title="Attention Is All You Need", author="Vaswani, A.", year="2017")]
        v2 = [self._e("vit_transformer", title="Attention is all you need", author="Vaswani, A.",
                      year="2017", doi="10.5555/3295222")]
        v3 = [self._e("vit_transformer", title="Attention is all you need", author="Vaswani, A.")]
        r = dedup_references({1: v1, 2: v2, 3: v3})
        self.assertEqual(r.n_before, 3)
        self.assertEqual(r.n_after, 1)
        (u,) = r.entries
        self.assertEqual(u["key"], "vit_transformer")
        self.assertEqual(set(u["fields"]), {"title", "author", "year", "doi"})
        self.assertEqual(r.key_map, {1: {"vaswani2017": "vit_transformer"}})
        row = entry_row("2310.12345", u)
        self.assertEqual(row["year"], 2017)
        self.assertEqual(row["doi"], "10.5555/3295222")

    def test_same_title_different_works_stay_apart(self):
        v1 = [self._e("a", title="Introduction to the theory of everything", year="2001"),
              self._e("b", title="Introduction to the theory of everything", year="2005")]
        self.assertEqual(dedup_references({1: v1}).n_after, 2)
        v2 = [self._e("c", title="Some long shared title words", doi="10.1234/x"),
              self._e("d", title="Some long shared title words", doi="10.1234/y")]
        self.assertEqual(dedup_references({1: v2}).n_after, 2)

    def test_doi_and_arxiv_merge_with_different_titles(self):
        v1 = [self._e("x1", title="Old title", doi="https://doi.org/10.1234/ABC")]
        v2 = [self._e("x2", title="Totally new title", doi="10.1234/abc"),
              self._e("y2", title="Preprint", journal="arXiv preprint arXiv:1706.03762")]
        v3 = [self._e("y3", title="Published", eprint="1706.03762", archiveprefix="arXiv")]
        r = dedup_references({1: v1, 2: v2, 3: v3})
        self.assertEqual(r.n_after, 2)
        self.assertEqual(r.key_map, {1: {"x1": "x2"}, 2: {"y2": "y3"}})

    def test_key_reused_for_different_work_gets_suffix(self):
        v1 = [self._e("smith", title="A first paper on graphs", year="2001")]
        v2 = [self._e("smith", title="Completely unrelated chemistry work", year="2019")]
        r = dedup_references({1: v1, 2: v2})
        self.assertEqual(sorted(e["key"] for e in r.entries), ["smith", "smith_2"])
        self.assertEqual(r.key_map, {1: {"smith": "smith_2"}})

    def test_duplicate_inside_version_keeps_most_cited_key(self):
        v1 = [self._e("k1", title="Graph neural networks survey", year="2020"),
              self._e("k2", title="Graph Neural Networks: survey", year="2020")]
        r = dedup_references({1: v1}, cite_counts={1: Counter({"k2": 3, "k1": 1})})
        self.assertEqual(r.n_after, 1)
        self.assertEqual(r.key_map, {1: {"k1": "k2"}})

    # ---------- version folder ----------
    def test_extract_version_sources(self):
        with tempfile.TemporaryDirectory() as tmp:
            v = Path(tmp)
            (v / "sec").mkdir()
            (v / "sec" / "Refs.bib").write_text(
                "@article{a, title={Cited one}}\n@article{b, title={Never cited}}", encoding="utf-8")
            tex = r"\cite{a} \bibliography{sec/refs}"
            r = extract_version_refs(v, tex)
            self.assertEqual(r.source, "bib")
            self.assertEqual([e["key"] for e in r.entries], ["a"])
            self.assertEqual(r.uncited_dropped, 1)
            self.assertEqual(extract_version_refs(v, tex, only_cited=False).n_raw, 2)
            r2 = extract_version_refs(v, r"\cite{a} \bibliography{other}")
            self.assertTrue(r2.bib_fallback)
        with tempfile.TemporaryDirectory() as tmp:
            v = Path(tmp)
            self.assertEqual(extract_version_refs(v, r"\cite{a}\bibliography{refs}").source, "missing_bbl")
            self.assertEqual(extract_version_refs(v, "").source, "none")
            (v / "main.bbl").write_text("\\begin{thebibliography}{1}\\bibitem{a} A.~B. Doe, Title words here, "
                                        "J. Phys. 1 (2000) 1.\\end{thebibliography}", encoding="utf-8")
            self.assertEqual(extract_version_refs(v, r"\cite{a}\bibliography{refs}").source, "bbl")
            r3 = extract_version_refs(v, r"\begin{thebibliography}{1}\bibitem{z} Z. Z, A title, 2001."
                                         r"\end{thebibliography}")
            self.assertEqual(r3.source, "bibitem")

    def test_bibitem_other_layouts(self):
        es = parse_bibitems(r"""\bibitem{MMD}
Liu, J. W. H.: Modification of the Minimum-Degree Algorithm by Multiple Elimination. ACM Trans. Math.
Softw. \textbf{11}(2), 141--153 (1985). \doi{10.1145/214393.214403}
\bibitem[ACOGM20]{ayre}
Peter Ayre, Amin Coja-Oghlan, Pu~Gao, and No{\"e}la M{\"u}ller, \emph{The satisfiability threshold
for random linear equations}, Combinatorica \textbf{40} (2020), no.~2, 179--235.
\bibitem{ayre2}
\bysame, \emph{Another paper by the same people}, Combinatorica \textbf{41} (2021).""")
        a, b, c = (e["fields"] for e in es)
        self.assertTrue(a["title"].startswith("Modification of the Minimum-Degree"))
        self.assertEqual(split_bib_authors(a["author"]), ["J. W. H. Liu"])
        self.assertEqual((a["year"], a["doi"]), ("1985", "10.1145/214393.214403"))
        self.assertEqual(b["title"], "The satisfiability threshold for random linear equations")
        self.assertEqual(split_bib_authors(b["author"])[-1], "Noëla Müller")
        self.assertEqual(b["journal"], "Combinatorica")
        self.assertEqual(c["author"], b["author"])

    def test_math_and_macro_params(self):
        self.assertEqual(latex_to_text(r"M\(^3\)CoT"), "M$^3$CoT")
        self.assertNotIn("#1", cited_keys(r"\newcommand{\mc}[1]{\cite{#1}} \cite{a}"))

    def test_entry_to_bibtex_roundtrip(self):
        e = {"key": "k", "type": "article", "fields": {"title": "T {X}", "_raw": "x", "year": "2020"}}
        (back,) = parse_bib(entry_to_bibtex(e))
        self.assertEqual(back["fields"], {"title": "T {X}", "year": "2020"})

    # ---------- added: .bib edge cases ----------
    def test_parse_bib_nested_braces_preamble_quotes(self):
        src = r'''@preamble{ "\newcommand{\noopsort}[1]{}" }
        @Book{n1, title = {Outer {Inner {Deep}} text}, author = "A {B} and C \"{o}",
              note = "say {"}hi{"}", pages = 1 # "--" # 9, publisher = {X}, }
        @comment{ @article{hidden, title={x}} }
        @article{n2, title = {Second}}'''
        es = {e["key"]: e for e in parse_bib(src)}
        self.assertEqual(sorted(es), ["n1", "n2"])
        f = es["n1"]["fields"]
        self.assertEqual(es["n1"]["type"], "book")
        self.assertEqual(f["title"], "Outer {Inner {Deep}} text")
        self.assertEqual(f["author"], r'A {B} and C \"{o}')
        self.assertEqual(f["note"], 'say {"}hi{"}')
        self.assertEqual(f["pages"], "1--9")
        self.assertEqual(latex_to_text(f["title"]), "Outer Inner Deep text")

    # ---------- added: \bibitem / .bbl ----------
    def test_two_thebibliography_blocks_and_bbl_same_function(self):
        text = (r"\begin{thebibliography}{9}\bibitem{a} A.~B. Doe, First title words here, J. Phys. 1 (2000) 1."
                r"\end{thebibliography} appendix \begin{thebibliography}{9}"
                r"\bibitem{b} C.~D. Roe, Second title words here, J. Chem. 2 (2001) 2.\end{thebibliography}")
        self.assertEqual([e["key"] for e in parse_bibitems(text)], ["a", "b"])
        # a classic .bbl goes through the same function as \bibitem in a .tex
        self.assertEqual(parse_bbl(text), parse_bibitems(text))

    def test_biblatex_bbl_multiline_and_legacy_names(self):
        bbl = r"""\entry{k1}{article}{}
      \name{author}{2}{}{%
        {{hash=aa}{%
           family={M{\"u}ller},
           familyi={M\bibinitperiod},
           given={Jan},
           giveni={J\bibinitperiod}}}%
        {{hash=bb}{%
           family={{OpenAI}},
           familyi={O\bibinitperiod}}}%
      }
      \field{journaltitle}{Nature}
      \field{title}{A title}
      \field{year}{2020}
      \verb{eprint}
      \verb 2001.00001
      \endverb
      \field{eprinttype}{arxiv}
    \endentry
    \entry{k2}{book}{}
      \name{author}{1}{}{%
        {{hash=cc}{Knuth}{K\bibinitperiod}{Donald}{D\bibinitperiod}{}{}{}{}}%
      }
      \field{title}{Old format}
      \field{date}{1984-05-01}
    \endentry"""
        a, b = parse_bbl(bbl)
        self.assertEqual(split_bib_authors(a["fields"]["author"]), ["Jan Müller", "OpenAI"])
        self.assertEqual(a["fields"]["journal"], "Nature")
        self.assertEqual(find_arxiv_id(a["fields"]), "2001.00001")
        self.assertEqual(split_bib_authors(b["fields"]["author"]), ["Donald Knuth"])
        self.assertEqual(entry_row("x", b)["year"], 1984)

    # ---------- added: cite commands ----------
    def test_rename_keeps_unknown_keys_optional_args_and_star(self):
        t = r"\citet*{old1, keep} \citeauthor{old2} \nocite{*} \cite[p.~3]{keep} \citep[see][p.~5]{old1,old2}"
        out = rename_cites(t, {"old1": "new", "old2": "new"})
        self.assertEqual(out, r"\citet*{new,keep} \citeauthor{new} \nocite{*} \cite[p.~3]{keep} "
                              r"\citep[see][p.~5]{new}")
        self.assertEqual(set(cited_keys(out)), {"new", "keep", "*"})
        # a command that needs no change is left byte-for-byte, even with odd spacing
        self.assertEqual(rename_cites(r"\cite{ a ,b }", {"zzz": "y"}), r"\cite{ a ,b }")
        # a cite inside an optional argument, and a macro definition that wraps \cite
        self.assertEqual(rename_cites(r"\citet[see also \cite{old1}]{old2}", {"old1": "n1", "old2": "n2"}),
                         r"\citet[see also \cite{n1}]{n2}")
        macro = r"\newcommand\mycite[1]{\citep{#1}} \mycite{old1}"
        self.assertEqual(rename_cites(macro, {"old1": "n1"}), r"\newcommand\mycite[1]{\citep{#1}} \mycite{n1}")

    # ---------- added: version folder ----------
    def test_only_cited_filter_nocite_star_and_unresolved(self):
        with tempfile.TemporaryDirectory() as tmp:
            v = Path(tmp)
            (v / "refs.bib").write_text("@article{a, title={One}}\n@article{b, title={Two}}\n"
                                        "@article{c, title={Three}}", encoding="utf-8")
            r = extract_version_refs(v, r"\cite{a,ghost} \bibliography{refs}")
            self.assertEqual((r.source, r.n_raw, len(r.entries), r.uncited_dropped), ("bib", 3, 1, 2))
            self.assertEqual(r.unresolved_cites, 1)                     # 'ghost' has no entry
            self.assertEqual(len(extract_version_refs(v, r"\nocite{*} \cite{a} \bibliography{refs}").entries), 3)
            self.assertEqual(len(extract_version_refs(v, r"\bibliography{refs}").entries), 3)   # no \cite at all
            # \addbibresource with extension and different case
            self.assertEqual(extract_version_refs(v, r"\cite{b} \addbibresource{REFS.bib}").source, "bib")

    def test_fallback_tex_reads_folder_and_strips_comments(self):
        with tempfile.TemporaryDirectory() as tmp:
            v = Path(tmp)
            (v / "main.tex").write_text("% \\bibliography{old}\n\\cite{a} % \\cite{b}\n\\bibliography{refs}\n",
                                        encoding="utf-8")
            (v / "refs.bib").write_text("@article{a, title={One}}\n@article{b, title={Two}}", encoding="utf-8")
            r = extract_version_refs(v, None)
            self.assertEqual(r.declared, ["refs"])
            self.assertEqual([e["key"] for e in r.entries], ["a"])

    # ---------- added: dedup rules ----------
    def test_key_rule_needs_both_titles_and_a_non_numeric_key(self):
        t1, t2 = "Deep learning for graphs: a survey", "Deep learning for graphs, a comprehensive survey"
        r = dedup_references({1: [self._e("smith20", title=t1, year="2020")],
                              2: [self._e("smith20", title=t2, year="2021")]})
        self.assertEqual(r.n_after, 1)                                  # same key + 6/7 shared words
        self.assertEqual(r.merges_by_rule["same_key_across_versions"], 1)
        self.assertEqual(r.merge_log, [{"rule": "same_key_across_versions", "version_a": 1,
                                        "key_a": "smith20", "version_b": 2, "key_b": "smith20"}])
        # one side has no title: not merged by the key; the older entry gets a suffix
        r = dedup_references({1: [self._e("web", url="https://a.example")],
                              2: [self._e("web", title="Some web page title here")]})
        self.assertEqual(sorted(e["key"] for e in r.entries), ["web", "web_2"])
        # titles too different: not merged
        r = dedup_references({1: [self._e("x", title="A first paper on graphs")],
                              2: [self._e("x", title="Completely unrelated chemistry work")]})
        self.assertEqual(r.n_after, 2)
        # purely numeric key: never merged by the key rule, whatever the titles
        r = dedup_references({1: [self._e("15", title=t1)], 2: [self._e("15", title=t2)]})
        self.assertEqual(r.n_after, 2)
        self.assertNotIn("same_key_across_versions", r.merges_by_rule)
        # ... but a renumbered list still merges through the content rules
        r = dedup_references({1: [self._e("15", title=t1, year="2020")], 2: [self._e("16", title=t1)]})
        self.assertEqual([e["key"] for e in r.entries], ["16"])
        self.assertEqual(r.key_map, {1: {"15": "16"}})

    def test_title_rule_does_not_chain_two_different_years(self):
        t = "Introduction to the theory of everything"
        r = dedup_references({1: [self._e("a", title=t, year="2001"), self._e("b", title=t),
                                  self._e("c", title=t, year="2005")]})
        self.assertEqual(r.n_after, 2)
        # different arXiv ids are never merged, even with the same title and year
        r = dedup_references({1: [self._e("p", title=t, year="2020", eprint="2001.00001"),
                                  self._e("q", title=t, year="2020", eprint="2001.00002")]})
        self.assertEqual(r.n_after, 2)

    def test_newer_version_wins_on_conflicting_fields(self):
        r = dedup_references({1: [self._e("k", title="Same long title here", year="2019", pages="1--2")],
                              2: [self._e("k", title="Same long title here", year="2019", pages="10--20",
                                          journal="J. New")]})
        (u,) = r.entries
        self.assertEqual((u["fields"]["pages"], u["fields"]["journal"]), ("10--20", "J. New"))
        self.assertEqual(u["versions"], [1, 2])
        self.assertEqual(r.key_map, {})

    # ---------- added: tables ----------
    def test_bibtex_row_has_the_nine_columns(self):
        e = {"key": "k", "type": "inproceedings",
             "fields": {"title": "{A} Title", "author": "Doe, John and Roe, Rita", "booktitle": "NeurIPS"}}
        row = entry_row("2605.07338", e)
        self.assertEqual(list(row), BIBTEX_COLUMNS)
        self.assertEqual(len(BIBTEX_COLUMNS), 9)
        self.assertEqual(row["authors"], ["John Doe", "Rita Roe"])
        self.assertIsNone(row["year"])
        self.assertIsNone(row["doi"])
        self.assertEqual(key_map_rows("2605.07338", {2: {"b": "c"}, 1: {"a": "c"}}),
                         [{"arxiv_id": "2605.07338", "version": 1, "old_key": "a", "new_key": "c"},
                          {"arxiv_id": "2605.07338", "version": 2, "old_key": "b", "new_key": "c"}])

    def test_process_paper_and_build_tables_end_to_end(self):
        global extract_version_refs
        bib = {1: "@article{vaswani2017, title={Attention Is All You Need}, author={Vaswani, A.}, year={2017}}",
               2: "@article{vit_transformer, title={Attention is all you need}, author={Vaswani, A.}, "
                  "year={2017}, doi={10.5555/3295222}}",
               3: "@article{vit_transformer, title={Attention is all you need}, author={Vaswani, A.}}"}
        tex = {1: r"As shown in \citep[see][]{vaswani2017}. \bibliography{refs}",
               2: r"As shown in \citep{vit_transformer}. \bibliography{refs}",
               3: r"As shown in \citet{vit_transformer}. \bibliography{refs}"}
        with tempfile.TemporaryDirectory() as tmp:
            raw, out = Path(tmp) / "raw", Path(tmp) / "out"
            for v in bib:
                d = raw / "2605-07338" / f"v{v}"
                d.mkdir(parents=True)
                (d / "main.tex").write_text(tex[v], encoding="utf-8")
                (d / "refs.bib").write_text(bib[v], encoding="utf-8")
            (raw / "2605-07339" / "v1").mkdir(parents=True)             # PDF-only version: empty folder
            (raw / "2605-07340" / "v1").mkdir(parents=True)             # this paper will raise
            (raw / "hep-th_9901001" / "v1").mkdir(parents=True)         # old-style id

            res = process_paper_bib("2605.07338", {v: (raw / "2605-07338" / f"v{v}", None) for v in bib})
            self.assertTrue({"rows", "key_map", "version_stats", "paper_stats"} <= set(res))
            self.assertEqual(res["key_map"], {1: {"vaswani2017": "vit_transformer"}})
            (row,) = res["rows"]
            self.assertEqual((row["citation_key"], row["year"], row["doi"]),
                             ("vit_transformer", 2017, "10.5555/3295222"))
            self.assertEqual((res["paper_stats"]["entries_before"], res["paper_stats"]["entries_after"],
                              res["paper_stats"]["renamed_keys"]), (3, 1, 1))
            # after renaming, version 1 cites no old key any more
            renamed = rename_cites(tex[1], res["key_map"][1])
            self.assertEqual(renamed, r"As shown in \citep[see][]{vit_transformer}. \bibliography{refs}")
            self.assertFalse(set(cited_keys(renamed)) & set(res["key_map"][1]))

            real = extract_version_refs

            def flaky(version_dir, tex=None, only_cited=None):
                if "2605-07340" in str(version_dir):
                    raise ValueError("broken paper")
                return real(version_dir, tex, only_cited)

            extract_version_refs = flaky
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    written, tables, errors = build_bibtex_tables(raw, out, progress_every=0)
            finally:
                extract_version_refs = real
            self.assertEqual(errors, [{"arxiv_id": "2605.07340", "error": "ValueError: broken paper"}])
            self.assertEqual([r["citation_key"] for r in tables["bibtex"]], ["vit_transformer"])
            self.assertEqual(tables["bib_key_map"], key_map_rows("2605.07338", res["key_map"]))
            stats = {(r["arxiv_id"], r["version"]): r["bib_source"] for r in tables["bib_version_stats"]}
            self.assertEqual(stats, {("2605.07338", 1): "bib", ("2605.07338", 2): "bib",
                                     ("2605.07338", 3): "bib", ("2605.07339", 1): "none",
                                     ("hep-th/9901001", 1): "none"})
            self.assertEqual(set(written), set(BIB_TABLES))
            self.assertTrue(all(p.is_file() and p.suffix in (".parquet", ".jsonl") for p in written.values()))
            with contextlib.redirect_stdout(io.StringIO()):
                _, limited, _ = build_bibtex_tables(raw, out, limit=1, progress_every=0)
                summary = bibtex_summary(tables, errors, 1.0)
            self.assertEqual(len(limited["bib_paper_stats"]), 1)
            self.assertEqual((summary["versions_by_source"]["bib"], summary["entries_before"],
                              summary["entries_after"], summary["renamed_keys"], summary["errors"]),
                             (3, 3, 1, 1, 1))


if RUN_TEST:
    _res = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(BibtexTests))
    assert _res.wasSuccessful(), "BibTeX tests failed"

## 14. Run on the crawled data (Colab with Drive mounted, or a local sample)

Runs only when `RUN_TEST` is on and `DATA_DIR` holds at least one version folder
(`<yymm-id>/vN/`). It processes the first `BIB_LIMIT` paper folders (set `BIB_LIMIT = None` for the
whole corpus), writes the tables to `PROJECT_ROOT/data/parsed/bibtex/` and prints the versions per
source label, the entries before and after deduplication, the renamed keys, the errors and the
run time.

In [ ]:
BIB_LIMIT = 500        # first N paper folders; set to None for the whole corpus


def _has_crawled_sources(data_dir):
    """True when at least one paper folder holds a version folder (v1, v2, ...)."""
    data_dir = Path(data_dir)
    return data_dir.is_dir() and any(_version_dirs(p) for p in data_dir.iterdir() if p.is_dir())


if RUN_TEST and "DATA_DIR" in globals() and _has_crawled_sources(DATA_DIR):
    BIB_OUT = Path(PROJECT_ROOT) / "data" / "parsed" / "bibtex"
    _t0 = time.perf_counter()
    _written, _tables, _errors = build_bibtex_tables(DATA_DIR, BIB_OUT, limit=BIB_LIMIT)
    BIB_SUMMARY = bibtex_summary(_tables, _errors, time.perf_counter() - _t0)
    print("written:", *[p.name for p in _written.values()])
elif RUN_TEST:
    print("[bibtex] no crawled sources under DATA_DIR: full run skipped")